In [1]:
import subprocess
subprocess.run(["pip", "install", "mlflow", "scikit-learn", "-q"])

import mlflow

mlflow.set_tracking_uri("http://localhost:5001")

print("✅ Connected to:", mlflow.get_tracking_uri())

✅ Connected to: http://localhost:5001


In [2]:
mlflow.set_experiment("researcher_abc")

print("✅ Experiment set: researcher_abc")

2026/03/15 14:44:19 INFO mlflow.tracking.fluent: Experiment with name 'researcher_abc' does not exist. Creating a new experiment.


✅ Experiment set: researcher_abc


In [3]:
with mlflow.start_run(run_name="test_run_1"):

    # log hyperparams
    mlflow.log_param("learning_rate", 0.01)
    mlflow.log_param("epochs",        10)
    mlflow.log_param("batch_size",    32)

    # simulate training metrics
    for epoch in range(10):
        mlflow.log_metric("accuracy", 0.7 + epoch * 0.025, step=epoch)
        mlflow.log_metric("loss",     0.5 - epoch * 0.04,  step=epoch)

    run_id = mlflow.active_run().info.run_id
    print(f"✅ Run logged! ID: {run_id}")

✅ Run logged! ID: a90c96f41587498fac445c49e742e8c6
🏃 View run test_run_1 at: http://localhost:5001/#/experiments/1/runs/a90c96f41587498fac445c49e742e8c6
🧪 View experiment at: http://localhost:5001/#/experiments/1


In [4]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

X, y = make_classification(n_samples=200, n_features=4, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

model = LogisticRegression().fit(X_train, y_train)
acc   = accuracy_score(y_test, model.predict(X_test))

with mlflow.start_run(run_name="logistic_regression_v1"):
    mlflow.log_param("model_type", "LogisticRegression")
    mlflow.log_metric("accuracy", acc)

    mlflow.sklearn.log_model(
        model,
        "model",
        registered_model_name="researcher_abc_model"
    )
    print(f"✅ Model registered! Accuracy: {acc:.4f}")

2026/03/15 14:44:50 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run logistic_regression_v1 at: http://localhost:5001/#/experiments/1/runs/716f801a36204ebab5298522e1e24020
🧪 View experiment at: http://localhost:5001/#/experiments/1


MlflowException: API request to endpoint /api/2.0/mlflow/logged-models failed with error code 404 != 200. Response body: '<!doctype html>
<html lang=en>
<title>404 Not Found</title>
<h1>Not Found</h1>
<p>The requested URL was not found on the server. If you entered the URL manually please check your spelling and try again.</p>
'

In [5]:
client = mlflow.MlflowClient("http://localhost:5001")

# list experiments
experiments = client.search_experiments()
print("📂 Experiments:")
for e in experiments:
    print(f"   - {e.name}")

# list registered models
models = client.search_registered_models()
print("📦 Registered models:")
for m in models:
    print(f"   - {m.name} ({len(m.latest_versions)} version(s))")

print("🎉 All good! Open http://localhost:5001 to see it in the UI")

📂 Experiments:
   - researcher_abc
   - Default
📦 Registered models:
🎉 All good! Open http://localhost:5001 to see it in the UI
